In [ ]:
import pandas as pd
import numpy as np
from func import clean_df
import re, string

# model imports
from keras.models import Sequential
from keras.layers import Dense, Dropout
import keras.layers as layers

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# NLP imports
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.preprocessing import OneHotEncoder

In [ ]:
data = pd.read_csv('./dataset/data.csv', quotechar='"', escapechar='\\') # read the dataset
data = clean_df(df=data, drop_columns=['text','date']) # clean the dataset

print(len(data))
print(data['subject'].unique())
data.sample(10)

In [ ]:
x_data_lst = ['title','subject','reporter','content']

In [ ]:
# encoding the labels

def tokeniseEmbed_and_oneHot(df: pd.DataFrame, tokenised_columns: list[str], oneHotEnc_columns: list[str]):

    def remove_punctuation(tokenised_text:str) -> list[str]:
        '''
        Remove punctuation from the tokenized text.
        Args:
            tokenised_text (str): The tokenized text from which punctuation needs to be removed.
        '''
        pattern = re.compile('[%s]' % re.escape(string.punctuation)) 
        new_sentence = []
        for token in tokenised_text: 
            new_token = pattern.sub(u'', token) # Replace by an empty string
            if not new_token == u'':
                new_sentence.append(new_token) # Append only tokens which are not empty
        return new_sentence

    def tokenise_columns(df: pd.DataFrame, tokenised_columns: list[str]):
        stop_words = set(stopwords.words('english'))
        lemmatizer = WordNetLemmatizer()
        
        for column in tokenised_columns:
            temp_col = []

            for text in df[column].values:
                filtered_tokens = []

                # Tokenize & remove punctuation from the text
                tokenised_text = remove_punctuation(word_tokenize(text))

                # Remove stop words
                for token in tokenised_text:
                    if token.lower() not in stop_words:
                        filtered_tokens.append(token.lower())

                # Apply lemmatization
                lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]

                temp_col.append(lemmatized_tokens)
            
            df[column] = pd.Series(temp_col)
        
        return df
    
    def embed_tokenised_text(df: pd.DataFrame, tokenised_columns: list[str]):
        '''
        Embed the tokenized text for each column using Word2Vec.
        Args:
            df (pd.DataFrame): The input dataframe containing the tokenized text.
            columns (list[str]): The list of column names to be embedded.
        '''
        for column in tokenised_columns:
            model = Word2Vec(sentences=df[column].values, vector_size=100, window=5, min_count=1, workers=4)
            df[column] = df[column].apply(lambda tokens: np.mean([model.wv[token] for token in tokens if token in model.wv], axis=0))
        return df
    
    def oneHotEncode_columns(df: pd.DataFrame, oneHotEnc_columns: list[str]):
        encoder = OneHotEncoder(sparse=False)
        for column in oneHotEnc_columns:
            encoded_data = encoder.fit_transform(df[[column]])
            encoded_df = pd.DataFrame(encoded_data, columns=[f"{column}_{cat}" for cat in encoder.categories_[0]])
            df = pd.concat([df, encoded_df], axis=1).drop(column, axis=1)
        return df
    
    df = tokenise_columns(df, tokenised_columns)
    df = embed_tokenised_text(df, tokenised_columns)
    df = oneHotEncode_columns(df, oneHotEnc_columns)
    return df


tkn_cols = ['content']
oneHot_cols = ['title','subject','reporter']

tokeniseEmbed_and_oneHot(data, tkn_cols, oneHot_cols)

data.info()

In [ ]:
data.sample(10)

In [ ]:
# split the dataset into training and testing sets

x_data = data[x_data_lst].values
y_data = data['label'].values

In [ ]:
# # model creation and training
# epochs = 10
# batch_size = 32

# model = Sequential([
#     layers.Input(shape=(len(x_data_lst),)),
#     Dense(128, activation='relu'),
#     Dropout(0.2),
#     Dense(64, activation='relu'),
#     Dropout(0.2),
#     Dense(1, activation='sigmoid')
# ])

# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# model.fit(x_data, y_data, epochs=epochs, batch_size=batch_size)